In [114]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('data/NPPE1_Preprocessing1.csv')
df.shape
df.head(1)

(4000, 14)

,CRIM,ZN,INDUS,POLINDEX,RM,AGE,DIS,HIGHWAYCOUNT,TAX,PTRATIO,IMM,BPL,PRICE,RIVERSIDE
0,1.026769,1.429034,7.8513,1.134216,6.0,42.0,5.251911,5,279.201277,20.689586,398.81196,10.461456,22.991633,NO


In [116]:
df.describe()

,CRIM,ZN,INDUS,POLINDEX,RM,AGE,DIS,HIGHWAYCOUNT,TAX,PTRATIO,IMM,BPL,PRICE
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,2.174158,12.734715,11.208752,1.331900,7.044500,66.005000,4.820784,8.464000,376.932036,19.051638,370.615373,12.326057,24.355923
std,2.475646,23.312649,6.827509,0.601024,1.240526,29.100923,2.174427,7.460169,150.270818,2.267293,67.687586,6.143060,8.633523
min,0.031093,0.000281,0.633388,0.405085,-1.000000,-2.000000,1.163082,1.000000,187.025099,12.641899,0.532980,1.986295,6.348266
25%,0.622728,0.464314,5.705110,0.864207,6.000000,41.000000,3.117734,4.000000,277.418380,17.634064,378.426323,7.514027,19.259094
50%,1.227945,1.068245,8.906823,1.218840,7.000000,74.000000,4.325631,6.000000,308.255685,19.386282,392.915025,11.188226,22.634566
75%,2.377738,20.323531,18.509019,1.694232,8.000000,93.000000,6.197419,7.000000,405.418015,20.809079,396.962724,16.092990,26.947036
max,12.619405,101.670740,30.367209,4.156570,11.000000,103.000000,14.045318,27.000000,713.028133,24.322091,400.780583,35.997281,53.090256


In [117]:
df['PRICE'].mean()

np.float64(24.355923220694248)

In [118]:
(df['RM'] >= 5).sum()

np.int64(3953)

In [119]:
df['PRICE'].sort_values(ascending=False)[:10].mean()

np.float64(52.36590175716407)

In [120]:
df['RM'].value_counts()

RM
 7.0     1733
 6.0      961
 8.0      894
 9.0      276
 5.0       47
 10.0      40
-1.0       40
 4.0        7
 11.0       2
Name: count, dtype: int64

In [121]:
(df['AGE'] < 0).sum()

np.int64(50)

In [122]:
df['RIVERSIDE'].value_counts()

RIVERSIDE
NO         3595
YES         317
UNKNOWN      88
Name: count, dtype: int64

In [123]:
((df['RIVERSIDE'] == 'YES') & ((df['AGE'] <= 50) & (df['AGE'] >= 0))).sum()

np.int64(44)

In [124]:
df['HIGHWAYCOUNT'].isin([6, 7, 8]).sum()

np.int64(1211)

In [125]:
(df['PRICE'] < 10).sum()
((df['PRICE'] >= 10) & (df['PRICE'] < 20)).sum()
((df['PRICE'] >= 20) & (df['PRICE'] < 30)).sum()
((df['PRICE'] >= 30) & (df['PRICE'] < 40)).sum()
(df['PRICE'] >= 40).sum()

np.int64(43)

np.int64(1158)

np.int64(2028)

np.int64(503)

np.int64(268)

In [126]:
num_cols = df.select_dtypes(include='number').columns.tolist()

summary = []

for col in num_cols:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  IQR = q3 - q1
  outliers = df[(df[col] < q1 - 1.5 * IQR) | (df[col] > q3 + 1.5 * IQR)]
  count = len(outliers)
  pct = (count / len(df)) * 100
  summary.append({
      'column': col,
      'outlier_count': count,
      'outlier_pct': pct
  })

pd.DataFrame(summary)

,column,outlier_count,outlier_pct
0,CRIM,542,13.550
1,ZN,359,8.975
2,INDUS,0,0.000
3,POLINDEX,60,1.500
4,RM,40,1.000
5,AGE,0,0.000
6,DIS,52,1.300
7,HIGHWAYCOUNT,662,16.550
8,TAX,716,17.900
9,PTRATIO,8,0.200


In [127]:
# df[df['RM'] < 0].loc[:, 'RM'] = np.nan

df.loc[df['RM'] == -1, 'RM'] = np.nan
df.loc[df['AGE'] < 0, 'AGE'] = np.nan
df.loc[df['RIVERSIDE'] == 'UNKNOWN', 'RIVERSIDE'] = np.nan

In [128]:
from sklearn.model_selection import train_test_split
X = df.drop('PRICE', axis=1)
y = df["PRICE"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

X_train.shape
X_test.shape
y_train.shape
y_test.shape

(2800, 13)

(1200, 13)

(2800,)

(1200,)

In [129]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train[['RM']] = imputer.fit_transform(X_train[['RM']])
X_test[['RM']] = imputer.transform(X_test[['RM']])

imputer = SimpleImputer(strategy='mean')
X_train[['AGE']] = imputer.fit_transform(X_train[['AGE']])
X_test[['AGE']] = imputer.transform(X_test[['AGE']])

imputer = SimpleImputer(strategy='most_frequent')
X_train[['RIVERSIDE']] = imputer.fit_transform(X_train[['RIVERSIDE']])
X_test[['RIVERSIDE']] = imputer.transform(X_test[['RIVERSIDE']])

In [130]:
min_max_cols = X_train.drop(['INDUS', 'RIVERSIDE'], axis=1).columns.tolist()
print(len(min_max_cols))

11


In [131]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scalar = MinMaxScaler()
X_train[min_max_cols] = scalar.fit_transform(X_train[min_max_cols])
X_test[min_max_cols] = scalar.transform(X_test[min_max_cols])

scalar = StandardScaler()
X_train[['INDUS']] = scalar.fit_transform(X_train[['INDUS']])
X_test[['INDUS']] = scalar.transform(X_test[['INDUS']])

In [138]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
X_train[encoder.get_feature_names_out()] = encoder.fit_transform(X_train[['RIVERSIDE']])
X_test[encoder.get_feature_names_out()] = encoder.transform(X_test[['RIVERSIDE']])

X_train = X_train.drop('RIVERSIDE', axis=1)
X_test = X_test.drop('RIVERSIDE', axis=1)

In [140]:
X_train.shape

(2800, 14)